In [ ]:
# In[1]:

import os
import pandas as pd
import numpy as np

# Helper to process each file
def process_telemetry_file(filename, source, name_col_expected):
    if not os.path.exists(filename):
        return f"{filename} missing"
    df = pd.read_csv(filename)
    if df.shape[0] == 0:
        return f"{filename} empty"
    # Ensure required columns exist
    if not {'timestamp','cmdb_id','value', name_col_expected}.issubset(df.columns):
        return f"{filename} missing required columns"
    # parse timestamp as UTC per rules
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
    # normalize KPI name column to 'kpi_name'
    if name_col_expected != 'kpi_name':
        df = df.rename(columns={name_col_expected: 'kpi_name'})
    # group and compute aggregations
    grp = df.groupby(['cmdb_id','kpi_name'])['value']
    agg_df = grp.agg(count='count', mean='mean', min='min', max='max')
    p95 = grp.quantile(0.95).rename('p95')
    agg_df = agg_df.join(p95)
    agg_df = agg_df.reset_index()
    agg_df['source'] = source
    # reorder and format
    agg_df = agg_df[['source','cmdb_id','kpi_name','count','mean','min','max','p95']]
    # round numeric columns for compactness
    for c in ['mean','min','max','p95']:
        agg_df[c] = agg_df[c].astype(float).round(6)
    # sort by count desc and limit to top 50
    agg_df = agg_df.sort_values('count', ascending=False).head(50).reset_index(drop=True)
    return agg_df

# Process the three files
metric_summary = process_telemetry_file('metric.csv', 'metric', 'kpi_name')
trace_summary  = process_telemetry_file('trace.csv',  'trace',  'trace_name')
log_summary    = process_telemetry_file('log.csv',    'log',    'log_name')

# Display results compactly (use variable names as per rules)
metric_summary, trace_summary, log_summary

```
Out[1]:
```
summary = (
    "All three files (metric.csv, trace.csv, log.csv) were present and processed.\n"
    "Metrics (top groups): adservice has the largest metric group counts (cpu, diskio, error, latency-50/90). "
    "Notable metric values include productcatalogservice CPU (mean ~10.82, max ~19.82) and paymentservice memory (mean ~2.91e7, max ~3.27e7).\n"
    "Traces (top groups): checkoutservice dominates trace groups (duration_mean, duration_p95, row_count, etc.). "
    "Some checkoutservice spans show higher mean durations (e.g., from_frontendservice mean ~0.211s, p95 ~0.380s).\n"
    "Logs (top groups): log.row_count is highest for frontend (mean ~2417.84) and currencyservice (mean ~2130.52). "
    "log.error_count is 0.0 for the visible groups (no elevated error counts in the top summaries).\n"
    "If you want more detail on any of these top groups (more rows or specific component KPIs), tell me which one to expand."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(    source                cmdb_id    kpi_name  count          mean           min           max           p95
0   metric              adservice         cpu     25  1.347612e+00  7.189890e-01  4.074263e+00  3.193480e+00
1   metric              adservice      diskio     25  6.882881e+04  4.949737e+04  1.597894e+05  9.944515e+04
2   metric              adservice       error     25  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
3   metric              adservice  latency-50     25  3.028000e-03  2.990000e-03  3.080000e-03  3.058000e-03
4   metric              adservice  latency-90     25  4.653000e-03  4.600000e-03  4.743000e-03  4.707000e-03
..     ...                    ...         ...    ...           ...           ...           ...           ...
45  metric         paymentservice         mem     25  2.910711e+07  2.601311e+07  3.272507e+07  3.226817e+07
46  metric         paymentservice      socket     25  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00
47  metric         paymentservice    workload     25  8.948100e-01  6.377500e-01  1.227700e+00  1.194983e+00
48  metric  productcatalogservice         cpu     25  1.081952e+01  3.272895e+00  1.982082e+01  1.972029e+01
49  metric  productcatalogservice  latency-50     25  3.379700e-02  1.738000e-03  7.325400e-02  7.276900e-02

[50 rows x 8 columns],    source          cmdb_id                                  kpi_name  count        mean        min         max         p95
0   trace  checkoutservice  trace.from_checkoutservice.duration_mean     25    0.017819   0.003567    0.042527    0.037377
1   trace  checkoutservice   trace.from_checkoutservice.duration_p95     25    0.048707   0.005826    0.096269    0.095245
2   trace  checkoutservice     trace.from_checkoutservice.error_rate     25    0.000000   0.000000    0.000000    0.000000
3   trace  checkoutservice      trace.from_checkoutservice.row_count     25  219.560000  30.000000  345.000000  300.600000
4   trace  checkoutservice  trace.from_frontendservice.duration_mean     25    0.211201   0.093373    0.415608    0.380378
..    ...              ...                                       ...    ...         ...        ...         ...         ...
45  trace     emailservice   trace.from_checkoutservice.duration_p95     25    0.000456   0.000331    0.000738    0.000646
46  trace     emailservice     trace.from_checkoutservice.error_rate     25    0.000000   0.000000    0.000000    0.000000
47  trace     emailservice      trace.from_checkoutservice.row_count     25   25.960000   3.000000   40.000000   36.000000
48  trace     emailservice             trace.from_root.duration_mean     25    0.000047   0.000039    0.000066    0.000057
49  trace     emailservice              trace.from_root.duration_p95     25    0.000071   0.000046    0.000138    0.000095

[50 rows x 8 columns],    source                cmdb_id         kpi_name  count     mean    min     max     p95
0     log              adservice  log.error_count     25     0.00    0.0     0.0     0.0
1     log              adservice    log.row_count     25   406.04   48.0   466.0   463.6
2     log            cartservice  log.error_count     25     0.00    0.0     0.0     0.0
3     log            cartservice    log.row_count     25   699.64   88.0   816.0   793.8
4     log        checkoutservice  log.error_count     25     0.00    0.0     0.0     0.0
5     log        checkoutservice    log.row_count     25    77.88    9.0   120.0   108.0
6     log        currencyservice  log.error_count     25     0.00    0.0     0.0     0.0
7     log        currencyservice    log.row_count     25  2130.52  260.0  2520.0  2494.4
8     log           emailservice  log.error_count     25     0.00    0.0     0.0     0.0
9     log           emailservice    log.row_count     25    25.96    3.0    40.0    36.0
10    log               frontend  log.error_count     25     0.00    0.0     0.0     0.0
11    log               frontend    log.row_count     25  2417.84  297.0  2835.0  2774.2
12    log         paymentservice  log.error_count     25     0.00    0.0     0.0     0.0
13    log         paymentservice    log.row_count     25    51.92    6.0    80.0    72.0
14    log  recommendationservice  log.error_count     25     0.00    0.0     0.0     0.0
15    log  recommendationservice    log.row_count     25   510.64   63.0   583.0   580.6
16    log        shippingservice    log.row_count     25   413.04   52.0   504.0   489.2
17    log        shippingservice  log.error_count     25     0.00    0.0     0.0     0.0
18    log                  redis    log.row_count      4     5.00    5.0     5.0     5.0
19    log                  redis  log.error_count      4     0.00    0.0     0.0     0.0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import os
import pandas as pd

# Candidate services
candidates = ["checkoutservice","currencyservice","emailservice","productcatalogservice","recommendationservice"]

# Load metric.csv
fname = "metric.csv"
if not os.path.exists(fname):
    result = "no metric data for candidate services in window"
else:
    df_metric = pd.read_csv(fname)
    if df_metric.shape[0] == 0:
        result = "no metric data for candidate services in window"
    else:
        # Ensure required columns
        required = {'timestamp','cmdb_id','kpi_name','value'}
        if not required.issubset(df_metric.columns):
            result = "no metric data for candidate services in window"
        else:
            # Parse timestamps in UTC
            df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

            # Compute global P95 and P05 per full-series group BEFORE filtering
            grp = df_metric.groupby(['cmdb_id','kpi_name'])['value']
            p95 = grp.quantile(0.95).rename('global_p95')
            p05 = grp.quantile(0.05).rename('global_p05')
            thresholds = pd.concat([p95, p05], axis=1).reset_index()

            # Define incident window (UTC)
            window_start = pd.to_datetime("2024-01-16 07:48:53", utc=True)
            window_end   = pd.to_datetime("2024-01-16 08:18:53", utc=True)

            # Filter rows by candidate services and window (inclusive)
            df_window = df_metric[
                (df_metric['cmdb_id'].isin(candidates)) &
                (df_metric['timestamp'] >= window_start) &
                (df_metric['timestamp'] <= window_end)
            ].copy()

            if df_window.shape[0] == 0:
                result = "no metric data for candidate services in window"
            else:
                # Merge thresholds into windowed data
                df_w = df_window.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

                # Mark anomalies based on global thresholds
                df_w['is_high'] = df_w['value'] >= df_w['global_p95']
                df_w['is_low']  = df_w['value'] <= df_w['global_p05']
                df_w['is_any']  = df_w['is_high'] | df_w['is_low']

                # Aggregations per group in window
                aggs = df_w.groupby(['cmdb_id','kpi_name']).agg(
                    total_points_in_window = ('value','count'),
                    mean_value_in_window  = ('value','mean'),
                    max_value_in_window   = ('value','max'),
                    anomaly_count_high    = ('is_high','sum'),
                    anomaly_count_low     = ('is_low','sum'),
                ).reset_index()

                # Earliest anomaly timestamp per group (if any)
                earliest = (df_w[df_w['is_any']]
                            .groupby(['cmdb_id','kpi_name'])['timestamp']
                            .min()
                            .rename('earliest_anomaly_timestamp')
                            .reset_index())

                # Join earliest timestamps and global thresholds (p95/p05)
                res = aggs.merge(earliest, on=['cmdb_id','kpi_name'], how='left')
                res = res.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

                # Format earliest timestamp as ISO UTC string, keep nulls as NaT/None
                if 'earliest_anomaly_timestamp' in res.columns:
                    res['earliest_anomaly_timestamp'] = pd.to_datetime(res['earliest_anomaly_timestamp'])
                    # keep timezone-aware strings, or NaT
                    res['earliest_anomaly_timestamp'] = res['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
                    # convert 'NaT' strings to None
                    res.loc[res['earliest_anomaly_timestamp'].isna(), 'earliest_anomaly_timestamp'] = None

                # Reorder and round numeric columns for compactness
                res = res[[
                    'cmdb_id','kpi_name','total_points_in_window','mean_value_in_window','max_value_in_window',
                    'global_p95','global_p05','anomaly_count_high','anomaly_count_low','earliest_anomaly_timestamp'
                ]]
                res[['mean_value_in_window','max_value_in_window','global_p95','global_p05']] = \
                    res[['mean_value_in_window','max_value_in_window','global_p95','global_p05']].astype(float).round(6)

                # Sort by total anomalies desc and limit to top 20
                res['total_anomalies'] = res['anomaly_count_high'] + res['anomaly_count_low']
                res = res.sort_values(['total_anomalies','total_points_in_window'], ascending=[False,False]).head(20)
                res = res.drop(columns=['total_anomalies']).reset_index(drop=True)

                result = res

# Display result variable
result

```
Out[2]:
```
All candidate services have metric data in the window. Key findings (top anomalies):

- Highest/severe anomalies (every point in window flagged):  
  - checkoutservice — socket (25/25 high, 25/25 low)  
  - currencyservice — error (25/25), socket (25/25)  
  - emailservice — diskio (25/25), socket (25/25)  
  - recommendationservice — diskio (25/25), socket (25/25)  
  These groups show all 25 points in the window equal to the global P95 and P05 thresholds (earliest anomaly at 2024-01-16 07:51:00+00:00).

- Other groups with some anomalies (small counts):  
  - latency, cpu, mem, workload KPIs for checkoutservice and currencyservice and productcatalogservice.socket show 2 high and 2 low anomaly points in the window. Earliest such anomalies are also around 2024-01-16 07:51–07:57 UTC depending on KPI.

Interpretation: socket/diskio/error metrics across multiple candidate services are uniformly at threshold levels during the incident window (strong signal). Several latency/cpu/memory/workload KPIs show a small number of outliers. Earliest detected anomalies are ≈ 07:51 UTC. Next steps: inspect logs/traces around 07:50–07:52 UTC for the listed services (especially socket and diskio metrics).

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  total_points_in_window  mean_value_in_window  max_value_in_window    global_p95    global_p05  anomaly_count_high  anomaly_count_low earliest_anomaly_timestamp
0         checkoutservice      socket                      25          1.000000e+01         1.000000e+01  1.000000e+01  1.000000e+01                  25                 25   2024-01-16 07:51:00+0000
1         currencyservice       error                      25          0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00                  25                 25   2024-01-16 07:51:00+0000
2         currencyservice      socket                      25          4.000000e+00         4.000000e+00  4.000000e+00  4.000000e+00                  25                 25   2024-01-16 07:51:00+0000
3            emailservice      diskio                      25          0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00                  25                 25   2024-01-16 07:51:00+0000
4            emailservice      socket                      25          3.000000e+00         3.000000e+00  3.000000e+00  3.000000e+00                  25                 25   2024-01-16 07:51:00+0000
5   recommendationservice      diskio                      25          0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00                  25                 25   2024-01-16 07:51:00+0000
6   recommendationservice      socket                      25          5.000000e+00         5.000000e+00  5.000000e+00  5.000000e+00                  25                 25   2024-01-16 07:51:00+0000
7            emailservice  latency-50                      25          3.018000e-03         3.078000e-03  3.070000e-03  3.000000e-03                   2                 12   2024-01-16 07:51:00+0000
8            emailservice  latency-90                      25          4.632000e-03         4.740000e-03  4.726000e-03  4.600000e-03                   2                 12   2024-01-16 07:51:00+0000
9   productcatalogservice      socket                      25          2.568326e+01         4.901667e+01  4.870333e+01  5.000000e+00                   2                 12   2024-01-16 07:51:00+0000
10        checkoutservice         cpu                      25          3.773010e-01         4.800830e-01  4.784640e-01  2.931560e-01                   2                  2   2024-01-16 07:51:00+0000
11        checkoutservice  latency-50                      25          1.870730e-01         3.192940e-01  2.966160e-01  1.156210e-01                   2                  2   2024-01-16 07:51:00+0000
12        checkoutservice  latency-90                      25          4.188990e-01         8.725010e-01  7.976890e-01  2.275410e-01                   2                  2   2024-01-16 07:51:00+0000
13        checkoutservice         mem                      25          1.076780e+07         1.125519e+07  1.113117e+07  1.042712e+07                   2                  2   2024-01-16 07:52:00+0000
14        checkoutservice    workload                      25          9.019690e-01         1.203217e+00  1.195850e+00  6.646330e-01                   2                  2   2024-01-16 07:57:00+0000
15        currencyservice         cpu                      25          2.568952e+01         2.970355e+01  2.895993e+01  2.186823e+01                   2                  2   2024-01-16 08:00:00+0000
16        currencyservice  latency-50                      25          4.179000e-03         4.659000e-03  4.582000e-03  3.845000e-03                   2                  2   2024-01-16 07:56:00+0000
17        currencyservice  latency-90                      25          4.094900e-02         6.872400e-02  6.865400e-02  1.044800e-02                   2                  2   2024-01-16 07:53:00+0000
18        currencyservice         mem                      25          4.374374e+07         4.472839e+07  4.429535e+07  4.285064e+07                   2                  2   2024-01-16 07:52:00+0000
19        currencyservice    workload                      25          7.443797e+01         8.530118e+01  8.361692e+01  6.474119e+01                   2                  2   2024-01-16 07:51:00+0000```
```

In [ ]:
# In[3]:

import os
import pandas as pd

# Candidate services (reuse variable name pattern)
candidates = ["checkoutservice","currencyservice","emailservice","productcatalogservice","recommendationservice"]

# Load trace.csv
fname = "trace.csv"
if not os.path.exists(fname):
    result = "no trace data for candidate services in window"
else:
    df_trace = pd.read_csv(fname)
    if df_trace.shape[0] == 0:
        result = "no trace data for candidate services in window"
    else:
        required = {'timestamp','cmdb_id','trace_name','value'}
        if not required.issubset(df_trace.columns):
            result = "no trace data for candidate services in window"
        else:
            # Parse timestamps as UTC
            df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

            # Compute global thresholds (P95 and P05) per full-series group BEFORE filtering
            grp_full = df_trace.groupby(['cmdb_id','trace_name'])['value']
            global_p95 = grp_full.quantile(0.95).rename('global_p95')
            global_p05 = grp_full.quantile(0.05).rename('global_p05')
            thresholds = pd.concat([global_p95, global_p05], axis=1).reset_index()

            # Define incident window (UTC)
            window_start = pd.to_datetime("2024-01-16 07:48:53", utc=True)
            window_end   = pd.to_datetime("2024-01-16 08:18:53", utc=True)

            # Filter rows by candidate services and window (inclusive)
            df_window = df_trace[
                (df_trace['cmdb_id'].isin(candidates)) &
                (df_trace['timestamp'] >= window_start) &
                (df_trace['timestamp'] <= window_end)
            ].copy()

            if df_window.shape[0] == 0:
                result = "no trace data for candidate services in window"
            else:
                # Merge global thresholds into windowed data
                df_w = df_window.merge(thresholds, on=['cmdb_id','trace_name'], how='left')

                # Mark anomalies relative to global thresholds
                df_w['is_high'] = df_w['value'] >= df_w['global_p95']
                df_w['is_low']  = df_w['value'] <= df_w['global_p05']
                df_w['is_any']  = df_w['is_high'] | df_w['is_low']

                # Aggregations per group in window
                aggs = df_w.groupby(['cmdb_id','trace_name']).agg(
                    total_points_in_window = ('value','count'),
                    mean_value_in_window  = ('value','mean'),
                    max_value_in_window   = ('value','max'),
                    anomaly_count_high    = ('is_high','sum'),
                    anomaly_count_low     = ('is_low','sum'),
                ).reset_index()

                # Earliest anomaly timestamp per group (if any)
                earliest = (df_w[df_w['is_any']]
                            .groupby(['cmdb_id','trace_name'])['timestamp']
                            .min()
                            .rename('earliest_anomaly_timestamp')
                            .reset_index())

                # Join earliest timestamps and global thresholds
                res = aggs.merge(earliest, on=['cmdb_id','trace_name'], how='left')
                res = res.merge(thresholds, on=['cmdb_id','trace_name'], how='left')

                # Format earliest timestamp as ISO UTC string with offset, keep None for missing
                if 'earliest_anomaly_timestamp' in res.columns:
                    res['earliest_anomaly_timestamp'] = pd.to_datetime(res['earliest_anomaly_timestamp'])
                    res['earliest_anomaly_timestamp'] = res['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
                    res.loc[res['earliest_anomaly_timestamp'].isna(), 'earliest_anomaly_timestamp'] = None

                # Round numeric columns for compactness
                for c in ['mean_value_in_window','max_value_in_window','global_p95','global_p05']:
                    if c in res.columns:
                        res[c] = pd.to_numeric(res[c], errors='coerce').round(6)

                # Sort by total anomalies desc and limit to top 20
                res['total_anomalies'] = res['anomaly_count_high'] + res['anomaly_count_low']
                res = res.sort_values(['total_anomalies','total_points_in_window'], ascending=[False,False]).head(20)
                res = res[[
                    'cmdb_id','trace_name','total_points_in_window','mean_value_in_window','max_value_in_window',
                    'global_p95','global_p05','anomaly_count_high','anomaly_count_low','earliest_anomaly_timestamp'
                ]].reset_index(drop=True)

                result = res

# Display the result variable (DataFrame or message)
result

```
Out[3]:
```
Summary (plain English):

- Trace data for the five candidate services is present in the incident window.
- All reported trace groups in the window are error_rate metrics and their values are uniformly 0 (mean 0, max 0).
- Because the full-series global P95 and P05 are both 0, every window point meets both >=P95 and <=P05, so anomaly_count_high and anomaly_count_low are 25 for each group. Earliest flagged timestamp is 2024-01-16 07:51:00 UTC.
- Interpretation: this is not a real error spike — it's an artifact of thresholds equaling the constant series value. In other words, trace error rates show no actual errors during the window.
- Next steps (suggestion): investigate non-error trace KPIs (durations) and the metric/log signals (socket/diskio) already flagged by the metric analysis around ~07:50–07:52 UTC.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                   trace_name  total_points_in_window  mean_value_in_window  max_value_in_window  global_p95  global_p05  anomaly_count_high  anomaly_count_low earliest_anomaly_timestamp
0         checkoutservice        trace.from_checkoutservice.error_rate                      25                   0.0                  0.0         0.0         0.0                  25                 25   2024-01-16 07:51:00+0000
1         checkoutservice        trace.from_frontendservice.error_rate                      25                   0.0                  0.0         0.0         0.0                  25                 25   2024-01-16 07:51:00+0000
2         checkoutservice                   trace.from_root.error_rate                      25                   0.0                  0.0         0.0         0.0                  25                 25   2024-01-16 07:51:00+0000
3         checkoutservice          trace.to_checkoutservice.error_rate                      25                   0.0                  0.0         0.0         0.0                  25                 25   2024-01-16 07:51:00+0000
4         checkoutservice          trace.to_currencyservice.error_rate                      25                   0.0                  0.0         0.0         0.0                  25                 25   2024-01-16 07:51:00+0000
5         checkoutservice             trace.to_emailservice.error_rate                      25                   0.0                  0.0         0.0         0.0                  25                 25   2024-01-16 07:51:00+0000
6         checkoutservice           trace.to_paymentservice.error_rate                      25                   0.0                  0.0         0.0         0.0                  25                 25   2024-01-16 07:51:00+0000
7         checkoutservice    trace.to_productcatalogservice.error_rate                      25                   0.0                  0.0         0.0         0.0                  25                 25   2024-01-16 07:51:00+0000
8         currencyservice        trace.from_checkoutservice.error_rate                      25                   0.0                  0.0         0.0         0.0                  25                 25   2024-01-16 07:51:00+0000
9         currencyservice        trace.from_frontendservice.error_rate                      25                   0.0                  0.0         0.0         0.0                  25                 25   2024-01-16 07:51:00+0000
10        currencyservice                   trace.from_root.error_rate                      25                   0.0                  0.0         0.0         0.0                  25                 25   2024-01-16 07:51:00+0000
11           emailservice        trace.from_checkoutservice.error_rate                      25                   0.0                  0.0         0.0         0.0                  25                 25   2024-01-16 07:51:00+0000
12           emailservice                   trace.from_root.error_rate                      25                   0.0                  0.0         0.0         0.0                  25                 25   2024-01-16 07:51:00+0000
13  productcatalogservice        trace.from_checkoutservice.error_rate                      25                   0.0                  0.0         0.0         0.0                  25                 25   2024-01-16 07:51:00+0000
14  productcatalogservice        trace.from_frontendservice.error_rate                      25                   0.0                  0.0         0.0         0.0                  25                 25   2024-01-16 07:51:00+0000
15  productcatalogservice  trace.from_recommendationservice.error_rate                      25                   0.0                  0.0         0.0         0.0                  25                 25   2024-01-16 07:51:00+0000
16  productcatalogservice                   trace.from_root.error_rate                      25                   0.0                  0.0         0.0         0.0                  25                 25   2024-01-16 07:51:00+0000
17  recommendationservice        trace.from_frontendservice.error_rate                      25                   0.0                  0.0         0.0         0.0                  25                 25   2024-01-16 07:51:00+0000
18  recommendationservice  trace.from_recommendationservice.error_rate                      25                   0.0                  0.0         0.0         0.0                  25                 25   2024-01-16 07:51:00+0000
19  recommendationservice                   trace.from_root.error_rate                      25                   0.0                  0.0         0.0         0.0                  25                 25   2024-01-16 07:51:00+0000```
```

In [ ]:
# In[4]:

import os
import pandas as pd
import numpy as np

# Reuse candidate services if present, else define
try:
    candidates
except NameError:
    candidates = ["checkoutservice","currencyservice","emailservice","productcatalogservice","recommendationservice"]

# Incident window (UTC)
window_start = pd.to_datetime("2024-01-16 07:48:53", utc=True)
window_end   = pd.to_datetime("2024-01-16 08:18:53", utc=True)

# --- Part A: process log.csv ---
log_fname = "log.csv"
if not os.path.exists(log_fname):
    log_result = "log.csv missing"
else:
    df_log = pd.read_csv(log_fname)
    if df_log.shape[0] == 0:
        log_result = "log.csv empty"
    else:
        required = {'timestamp','cmdb_id','log_name','value'}
        if not required.issubset(df_log.columns):
            log_result = "log.csv missing required columns"
        else:
            # Compute global thresholds (P95, P05) per full-series BEFORE filtering
            grp_full = df_log.groupby(['cmdb_id','log_name'])['value'].apply(pd.to_numeric, errors='coerce')
            # Convert series to DataFrame to compute quantiles per group reliably
            # Use groupby on original df to preserve numeric conversion
            df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')
            thresholds = (df_log
                          .groupby(['cmdb_id','log_name'])['value']
                          .agg(global_p95=lambda s: s.quantile(0.95),
                               global_p05=lambda s: s.quantile(0.05))
                          .reset_index())

            # Parse timestamps as UTC
            df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

            # Filter to candidate services and window (inclusive)
            df_log_win = df_log[
                (df_log['cmdb_id'].isin(candidates)) &
                (df_log['timestamp'] >= window_start) &
                (df_log['timestamp'] <= window_end)
            ].copy()

            if df_log_win.shape[0] == 0:
                log_result = "no log data for candidate services in window"
            else:
                # Merge thresholds
                df_lw = df_log_win.merge(thresholds, on=['cmdb_id','log_name'], how='left')

                # Determine anomalies
                df_lw['is_high'] = df_lw['value'] >= df_lw['global_p95']
                df_lw['is_low']  = df_lw['value'] <= df_lw['global_p05']
                df_lw['is_any']  = df_lw['is_high'] | df_lw['is_low']

                # Aggregate per group
                aggs = df_lw.groupby(['cmdb_id','log_name']).agg(
                    total_points_in_window = ('value','count'),
                    mean_value_in_window  = ('value','mean'),
                    max_value_in_window   = ('value','max'),
                    anomaly_count_high    = ('is_high','sum'),
                    anomaly_count_low     = ('is_low','sum'),
                ).reset_index()

                # Earliest anomaly timestamp per group (if any)
                earliest = (df_lw[df_lw['is_any']]
                            .groupby(['cmdb_id','log_name'])['timestamp']
                            .min()
                            .rename('earliest_anomaly_timestamp')
                            .reset_index())

                # Combine results
                log_res = aggs.merge(earliest, on=['cmdb_id','log_name'], how='left')
                log_res = log_res.merge(thresholds, on=['cmdb_id','log_name'], how='left')

                # Format earliest timestamp and round numeric columns
                log_res['earliest_anomaly_timestamp'] = pd.to_datetime(log_res['earliest_anomaly_timestamp'])
                log_res['earliest_anomaly_timestamp'] = log_res['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
                log_res.loc[log_res['earliest_anomaly_timestamp'].isna(), 'earliest_anomaly_timestamp'] = None

                for c in ['mean_value_in_window','max_value_in_window','global_p95','global_p05']:
                    log_res[c] = pd.to_numeric(log_res[c], errors='coerce').round(6)

                log_res['total_anomalies'] = log_res['anomaly_count_high'] + log_res['anomaly_count_low']
                log_res = log_res.sort_values(['total_anomalies','total_points_in_window'], ascending=[False,False]).head(20)
                log_result = log_res[[
                    'cmdb_id','log_name','total_points_in_window','mean_value_in_window','max_value_in_window',
                    'global_p95','global_p05','anomaly_count_high','anomaly_count_low','earliest_anomaly_timestamp'
                ]].reset_index(drop=True)

# --- Part B: process error_logs.csv ---
err_fname = "error_logs.csv"
if not os.path.exists(err_fname):
    error_logs_result = "error_logs.csv missing"
    error_counts = "error_logs.csv missing"
else:
    df_err = pd.read_csv(err_fname)
    if df_err.shape[0] == 0:
        error_logs_result = "error_logs.csv empty"
        error_counts = "error_logs.csv empty"
    else:
        required_err = {'timestamp','cmdb_id','message'}
        if not required_err.issubset(df_err.columns):
            error_logs_result = "error_logs.csv missing required columns"
            error_counts = "error_logs.csv missing required columns"
        else:
            # Parse timestamps as UTC
            df_err['timestamp'] = pd.to_datetime(df_err['timestamp'], unit='s', utc=True)

            # Filter to candidate services and window
            df_err_win = df_err[
                (df_err['cmdb_id'].isin(candidates)) &
                (df_err['timestamp'] >= window_start) &
                (df_err['timestamp'] <= window_end)
            ].copy()

            if df_err_win.shape[0] == 0:
                error_logs_result = "no error_logs for candidate services in window"
                error_counts = "no error_logs for candidate services in window"
            else:
                # Keywords (case-insensitive). 'timed out' and 'timed out' vs 'timeout' included as given.
                keywords = ['socket','timeout','timed out','connection','refused','broken','io','disk','diskio','error']

                # Lowercase message for matching
                df_err_win['message_lc'] = df_err_win['message'].astype(str).str.lower()

                # Count per keyword (a message can be counted for multiple keywords)
                counts = {}
                for kw in keywords:
                    # For phrases like 'timed out' keep as is
                    mask = df_err_win['message_lc'].str.contains(kw, na=False)
                    counts[kw] = int(mask.sum())

                # Total unique messages matching any keyword
                any_mask = np.column_stack([df_err_win['message_lc'].str.contains(kw, na=False) for kw in keywords]).any(axis=1)
                total_matching = int(any_mask.sum())

                # Prepare counts DataFrame
                error_counts = pd.DataFrame({
                    'keyword': list(counts.keys()),
                    'count': list(counts.values())
                })
                error_counts = error_counts.sort_values('count', ascending=False).reset_index(drop=True)
                # Add total matching as a separate one-row DataFrame for compactness
                total_row = pd.DataFrame({'keyword': ['__total_matching_messages__'], 'count': [total_matching]})
                error_counts = pd.concat([error_counts, total_row], ignore_index=True)

                # Top 20 filtered raw error_log rows sorted by timestamp ascending
                df_err_win_sorted = df_err_win.sort_values('timestamp', ascending=True).head(20).copy()
                # Format timestamp as string with UTC offset
                df_err_win_sorted['timestamp_utc'] = pd.to_datetime(df_err_win_sorted['timestamp']).dt.strftime('%Y-%m-%d %H:%M:%S%z')
                error_logs_result = df_err_win_sorted[['timestamp_utc','cmdb_id','message']].rename(columns={'timestamp_utc':'timestamp'})

# Final outputs: log_result (DataFrame or message), error_counts (DataFrame or message), error_logs_result (DataFrame or message)
log_result, error_counts, error_logs_result

```
Out[4]:
```
Summary (plain English):

- log.csv processed; error_logs.csv is empty (no error_log messages for candidate services in the incident window).

- log.error_count: for all candidate services shown (checkoutservice, currencyservice, emailservice, recommendationservice) the error_count series is 0.0 in the window. Global P95/P05 are 0, so every point is flagged by the threshold rule (artifact of constant-zero series). Earliest flagged timestamp: 2024-01-16 07:51:00 UTC.

- log.row_count (window groups with notable anomaly counts):
  - checkoutservice — mean 77.88, max 120, global_p95 108.0, global_p05 54.0, anomaly_count_high=3, anomaly_count_low=3, earliest anomaly 2024-01-16 07:51:00 UTC.
  - emailservice — mean 25.96, max 40, global_p95 36.0, global_p05 18.0, anomaly_count_high=3, anomaly_count_low=3, earliest 2024-01-16 07:51:00 UTC.
  - currencyservice — mean 2130.52, max 2520, global_p95 2494.4, global_p05 1866.6, anomaly_count_high=2, anomaly_count_low=2, earliest 2024-01-16 07:51:00 UTC.
  - recommendationservice — mean 510.64, max 583, global_p95 580.6, global_p05 457.8, anomaly_count_high=2, anomaly_count_low=2, earliest 2024-01-16 07:51:00 UTC.

- error_logs.csv: empty for candidate services in the window — no raw error messages to inspect.

Interpretation / next steps:
- The zero error_count series does not indicate actionable errors (threshold artifact). Focus on row_count anomalies (traffic spikes/dips) for the listed services around ~07:51 UTC. Correlate these with the metric and trace findings (socket/diskio signals and trace durations) near 07:50–07:52 UTC.

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id         log_name  total_points_in_window  mean_value_in_window  max_value_in_window  global_p95  global_p05  anomaly_count_high  anomaly_count_low earliest_anomaly_timestamp
0        checkoutservice  log.error_count                      25                  0.00                    0         0.0         0.0                  25                 25   2024-01-16 07:51:00+0000
1        currencyservice  log.error_count                      25                  0.00                    0         0.0         0.0                  25                 25   2024-01-16 07:51:00+0000
2           emailservice  log.error_count                      25                  0.00                    0         0.0         0.0                  25                 25   2024-01-16 07:51:00+0000
3  recommendationservice  log.error_count                      25                  0.00                    0         0.0         0.0                  25                 25   2024-01-16 07:51:00+0000
4        checkoutservice    log.row_count                      25                 77.88                  120       108.0        54.0                   3                  3   2024-01-16 07:51:00+0000
5           emailservice    log.row_count                      25                 25.96                   40        36.0        18.0                   3                  3   2024-01-16 07:51:00+0000
6        currencyservice    log.row_count                      25               2130.52                 2520      2494.4      1866.6                   2                  2   2024-01-16 07:51:00+0000
7  recommendationservice    log.row_count                      25                510.64                  583       580.6       457.8                   2                  2   2024-01-16 07:51:00+0000, 'error_logs.csv empty', 'error_logs.csv empty')```
```

In [ ]:
# In[5]:

import os
import pandas as pd
import numpy as np

# Reuse df_trace if already loaded in this session, otherwise load from CSV
if 'df_trace' not in globals():
    if not os.path.exists('trace.csv'):
        result = "no trace data for candidate services in window"
    else:
        df_trace = pd.read_csv('trace.csv')

# If df_trace exists, proceed
if 'df_trace' in globals():
    if df_trace.shape[0] == 0:
        result = "no trace data for candidate services in window"
    else:
        # Ensure required columns
        if not {'timestamp','cmdb_id','trace_name','value'}.issubset(df_trace.columns):
            result = "no trace data for candidate services in window"
        else:
            # Parse timestamps as UTC (do once)
            df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

            # Candidate services
            candidates = ["checkoutservice","currencyservice","emailservice","productcatalogservice","recommendationservice"]

            # Compute global P95 and P05 per full-series group BEFORE filtering
            thresholds = (df_trace
                          .groupby(['cmdb_id','trace_name'])['value']
                          .agg(global_p95=lambda s: s.quantile(0.95),
                               global_p05=lambda s: s.quantile(0.05))
                          .reset_index())

            # Define incident window (UTC)
            window_start = pd.to_datetime("2024-01-16 07:48:53", utc=True)
            window_end   = pd.to_datetime("2024-01-16 08:18:53", utc=True)

            # Filter rows by candidate services, window, and trace_name containing 'duration'
            df_window = df_trace[
                (df_trace['cmdb_id'].isin(candidates)) &
                (df_trace['timestamp'] >= window_start) &
                (df_trace['timestamp'] <= window_end) &
                (df_trace['trace_name'].str.contains('duration', case=False, na=False))
            ].copy()

            if df_window.shape[0] == 0:
                result = "no duration traces for candidate services in window"
            else:
                # Merge thresholds into windowed data
                df_w = df_window.merge(thresholds, on=['cmdb_id','trace_name'], how='left')

                # Mark anomalies relative to global thresholds
                df_w['is_high'] = df_w['value'] >= df_w['global_p95']
                df_w['is_low']  = df_w['value'] <= df_w['global_p05']
                df_w['is_any']  = df_w['is_high'] | df_w['is_low']

                # Aggregations per group in window
                aggs = df_w.groupby(['cmdb_id','trace_name']).agg(
                    total_points_in_window = ('value','count'),
                    mean_value_in_window  = ('value','mean'),
                    max_value_in_window   = ('value','max'),
                    anomaly_count_high    = ('is_high','sum'),
                    anomaly_count_low     = ('is_low','sum'),
                ).reset_index()

                # Earliest anomaly timestamp per group (if any)
                earliest = (df_w[df_w['is_any']]
                            .groupby(['cmdb_id','trace_name'])['timestamp']
                            .min()
                            .rename('earliest_anomaly_timestamp')
                            .reset_index())

                # Combine results and attach global thresholds
                res = aggs.merge(earliest, on=['cmdb_id','trace_name'], how='left')
                res = res.merge(thresholds, on=['cmdb_id','trace_name'], how='left')

                # Format earliest timestamp as ISO UTC string with offset, keep None for missing
                res['earliest_anomaly_timestamp'] = pd.to_datetime(res['earliest_anomaly_timestamp'])
                res['earliest_anomaly_timestamp'] = res['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
                res.loc[res['earliest_anomaly_timestamp'].isna(), 'earliest_anomaly_timestamp'] = None

                # Round numeric columns for compactness
                for c in ['mean_value_in_window','max_value_in_window','global_p95','global_p05']:
                    res[c] = pd.to_numeric(res[c], errors='coerce').round(6)

                # Sort by total anomalies desc and limit to top 20
                res['total_anomalies'] = res['anomaly_count_high'] + res['anomaly_count_low']
                res = res.sort_values(['total_anomalies','total_points_in_window'], ascending=[False,False]).head(20)

                # Select and order final columns
                result = res[[
                    'cmdb_id','trace_name','total_points_in_window','mean_value_in_window','max_value_in_window',
                    'global_p95','global_p05','anomaly_count_high','anomaly_count_low','earliest_anomaly_timestamp'
                ]].reset_index(drop=True)

# Display result (DataFrame or message)
result

```
Out[5]:
```
Summary (plain English):

- Duration traces for the candidate services are present in the incident window. All listed groups have 25 points in-window.

- Notable groups with the largest anomaly counts:
  - productcatalogservice — trace.from_frontendservice.duration_p95: 2 points >= global P95 and 9 points <= global P05 (earliest anomaly 2024-01-16 07:51:00 UTC).
  - emailservice — trace.from_root.duration_p95: 3 points >= global P95 and 2 points <= global P05 (earliest anomaly 2024-01-16 07:51:00 UTC).

- Checkoutservice shows multiple duration metrics (from/to and root) with small numbers of anomalies (typically 2 high and 2 low per trace_name). Earliest checkoutservice duration anomalies occur around 07:51–07:58 UTC depending on the trace.

- Other candidate services (currencyservice, productcatalogservice, recommendationservice) show a few duration anomalies as well (mostly 2 high / 2 low for many trace_name entries).

- Timing: earliest anomalies are clustered ~07:51 UTC (some checkoutservice traces earliest at 07:54 or 07:58 UTC).

Interpretation / recommended next steps:
- There are modest but consistent deviations in duration p95/mean for productcatalogservice and emailservice and several checkoutservice spans around ~07:51 UTC. These could reflect latency fluctuations rather than a full outage.
- Correlate these duration deviations with the metric findings (socket/diskio signals) and the log row_count anomalies around 07:50–07:52 UTC to determine root cause and scope.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                    trace_name  total_points_in_window  mean_value_in_window  max_value_in_window  global_p95  global_p05  anomaly_count_high  anomaly_count_low earliest_anomaly_timestamp
0   productcatalogservice       trace.from_frontendservice.duration_p95                      25              0.000023             0.000026    0.000025    0.000022                   2                  9   2024-01-16 07:51:00+0000
1            emailservice                  trace.from_root.duration_p95                      25              0.000071             0.000138    0.000095    0.000050                   3                  2   2024-01-16 07:51:00+0000
2         checkoutservice      trace.from_checkoutservice.duration_mean                      25              0.017819             0.042527    0.037377    0.003784                   2                  2   2024-01-16 07:51:00+0000
3         checkoutservice       trace.from_checkoutservice.duration_p95                      25              0.048707             0.096269    0.095245    0.006354                   2                  2   2024-01-16 07:51:00+0000
4         checkoutservice      trace.from_frontendservice.duration_mean                      25              0.211201             0.415608    0.380378    0.109225                   2                  2   2024-01-16 07:58:00+0000
5         checkoutservice       trace.from_frontendservice.duration_p95                      25              0.446666             1.154137    0.841265    0.185604                   2                  2   2024-01-16 07:51:00+0000
6         checkoutservice                 trace.from_root.duration_mean                      25              0.013334             0.022386    0.021489    0.005294                   2                  2   2024-01-16 07:54:00+0000
7         checkoutservice                  trace.from_root.duration_p95                      25              0.045885             0.080337    0.080096    0.010424                   2                  2   2024-01-16 07:54:00+0000
8         checkoutservice        trace.to_checkoutservice.duration_mean                      25              0.017819             0.042527    0.037377    0.003784                   2                  2   2024-01-16 07:51:00+0000
9         checkoutservice         trace.to_checkoutservice.duration_p95                      25              0.048707             0.096269    0.095245    0.006354                   2                  2   2024-01-16 07:51:00+0000
10        checkoutservice        trace.to_currencyservice.duration_mean                      25              0.000627             0.002527    0.002182    0.000130                   2                  2   2024-01-16 07:51:00+0000
11        checkoutservice         trace.to_currencyservice.duration_p95                      25              0.000221             0.000258    0.000256    0.000185                   2                  2   2024-01-16 07:51:00+0000
12        checkoutservice           trace.to_emailservice.duration_mean                      25              0.000290             0.000406    0.000347    0.000249                   2                  2   2024-01-16 07:51:00+0000
13        checkoutservice            trace.to_emailservice.duration_p95                      25              0.000456             0.000738    0.000646    0.000340                   2                  2   2024-01-16 07:53:00+0000
14        checkoutservice         trace.to_paymentservice.duration_mean                      25              0.000394             0.000635    0.000525    0.000312                   2                  2   2024-01-16 08:08:00+0000
15        checkoutservice          trace.to_paymentservice.duration_p95                      25              0.000725             0.001435    0.001311    0.000384                   2                  2   2024-01-16 07:51:00+0000
16        checkoutservice  trace.to_productcatalogservice.duration_mean                      25              0.000016             0.000038    0.000025    0.000010                   2                  2   2024-01-16 07:56:00+0000
17        checkoutservice   trace.to_productcatalogservice.duration_p95                      25              0.000027             0.000043    0.000042    0.000020                   2                  2   2024-01-16 07:51:00+0000
18        currencyservice      trace.from_checkoutservice.duration_mean                      25              0.000627             0.002527    0.002182    0.000130                   2                  2   2024-01-16 07:51:00+0000
19        currencyservice       trace.from_checkoutservice.duration_p95                      25              0.000221             0.000258    0.000256    0.000185                   2                  2   2024-01-16 07:51:00+0000```
```